# Experiment 9: SAELens synthetic rho-model comparison

This notebook compares VG-SAE with five official SAELens 6.47.0 training paths. Full mode uses [decoderesearch/synth-sae-bench-16k-v1](https://huggingface.co/decoderesearch/synth-sae-bench-16k-v1); FAST mode builds a small official `SyntheticModel` and executes all 6 architectures at 2 controls each.

Fairness contract:

- every run receives a fresh official `SyntheticActivationIterator`, not a repeated finite cache;
- initialization, train-only scale calibration, training, and evaluation use distinct recorded seeds;
- the training RNG is reset after model initialization, giving every method the same fresh stream;
- one common `expected_average_only_in` scalar is used by every method;
- evaluation uses the registered inference export, decoder-norm folding, and official `eval_sae_on_synthetic_data`;
- official hard L0/recovery/classification metrics are primary, while VG posterior rho and expected reconstruction are separate.

Target: decoderesearch/SAELens v6.47.0, commit 8be14080485952f729ed58d674bcddf9778e0aa4; benchmark snapshot b2efd8b919ae46d6d487c73d46db5ee52813621d. Full defaults are an exploratory 256k-training/100k-evaluation protocol, not the official 200M-sample leaderboard recipe.


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import sys
import time

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "outputs" / ".matplotlib"))
REQUESTED_DEVICE = os.environ.get("VGSAE_DEVICE", "cpu")
if REQUESTED_DEVICE == "cpu":
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from huggingface_hub import snapshot_download

import sae_lens
from sae_lens import (
    BatchTopKTrainingSAE,
    BatchTopKTrainingSAEConfig,
    GatedTrainingSAE,
    GatedTrainingSAEConfig,
    JumpReLUTrainingSAE,
    JumpReLUTrainingSAEConfig,
    SAETrainer,
    StandardTrainingSAE,
    StandardTrainingSAEConfig,
    TopKTrainingSAE,
    TopKTrainingSAEConfig,
)
from sae_lens.config import SAETrainerConfig
from sae_lens.evals import ExplainedVarianceCalculator
from sae_lens.synthetic import (
    ConstantFiringProbabilityConfig,
    SyntheticActivationIterator,
    SyntheticModel,
    SyntheticModelConfig,
    eval_sae_on_synthetic_data,
)
from sae_lens.training.activation_scaler import ActivationScaler as OfficialActivationScaler
from sae_lens.util import temporary_seed

from src.sae_baselines import to_inference_sae
from src.saelens_data import ActivationScale
from src.saelens_vg import VGSAE, VGSAETrainer, VGTrainingSAE, VGTrainingSAEConfig

SAELENS_VERSION = "6.47.0"
SAELENS_COMMIT = "8be14080485952f729ed58d674bcddf9778e0aa4"
assert sae_lens.__version__ == SAELENS_VERSION, sae_lens.__version__
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
DEVICE = torch.device(REQUESTED_DEVICE)
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "notebooks" / "exp09_saelens_synthetic_v647"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Configuration

Set `VGSAE_NOTEBOOK_FAST_DEV_RUN=0` for the official 16k-feature generator. Full-mode sample counts, batches, seeds, and control grids can be overridden with environment variables. `VGSAE_BETA_MODE` accepts fixed or learned; profiled beta is intentionally excluded.


In [ ]:
def env_values(name, default, cast=float):
    raw = os.environ.get(name)
    return default if raw is None else [cast(value) for value in raw.split(",")]


FAST_DEV_RUN = bool(int(os.environ.get("VGSAE_NOTEBOOK_FAST_DEV_RUN", "1")))
RUN_EXPERIMENT = bool(int(os.environ.get("VGSAE_RUN_EXPERIMENT", "1")))
VG_BETA_MODE = os.environ.get("VGSAE_BETA_MODE", "fixed")
if VG_BETA_MODE not in {"fixed", "learned"}:
    raise ValueError("VGSAE_BETA_MODE must be 'fixed' or 'learned'.")

BENCHMARK_MODEL_ID = "decoderesearch/synth-sae-bench-16k-v1"
BENCHMARK_REVISION = "b2efd8b919ae46d6d487c73d46db5ee52813621d"
scratch_support_density, beta = 0.10, 1.0
data_model_seed_offset = 10_000
calibration_seed_offset = 20_000
train_seed_offset = 30_000
eval_seed_offset = 40_000
model_seed_offset = 50_000

if FAST_DEV_RUN:
    hidden_dim, n_true_features, d_sae = 16, 32, 32
    seeds = [0]
    batch_size, total_training_samples = 64, 256
    calibration_samples, eval_samples, eval_batch_size = 256, 256, 64
    gamma_values = [0.0, 1.0]
    l1_values = [1.0e-3, 1.0e-2]
    topk_values = [2, 4]
    jumprelu_values = [1.0e-3, 1.0e-2]
else:
    hidden_dim, n_true_features, d_sae = 768, 16_384, 4_096
    seeds = env_values("VGSAE_SEEDS", list(range(5)), int)
    batch_size = int(os.environ.get("VGSAE_BATCH_SIZE", "256"))
    total_training_samples = int(os.environ.get("VGSAE_TRAINING_SAMPLES", "256000"))
    calibration_samples = int(os.environ.get("VGSAE_CALIBRATION_SAMPLES", "65536"))
    eval_samples = int(os.environ.get("VGSAE_EVAL_SAMPLES", "100000"))
    eval_batch_size = int(os.environ.get("VGSAE_EVAL_BATCH_SIZE", str(batch_size)))
    gamma_values = env_values(
        "VGSAE_GAMMAS", [-0.5, 0.0, 0.5, 1.0, 2.0, 4.0, 8.0]
    )
    l1_values = env_values(
        "VGSAE_L1_VALUES",
        [1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2],
    )
    topk_values = env_values("VGSAE_TOPK_VALUES", [15, 25, 35, 45], int)
    jumprelu_values = env_values("VGSAE_JUMPRELU_VALUES", l1_values)

if total_training_samples % batch_size:
    raise ValueError("training samples must be divisible by batch size")
calibration_batch_size = min(batch_size, calibration_samples)
if calibration_samples % calibration_batch_size:
    raise ValueError("calibration samples must be divisible by calibration batch size")
learning_rate = float(os.environ.get("VGSAE_LEARNING_RATE", "3e-4"))
dead_feature_window = 2 if FAST_DEV_RUN else int(
    os.environ.get("VGSAE_DEAD_FEATURE_WINDOW", "100")
)


## Fresh official streams and common scale

SAELens' synthetic iterator uses the global Torch RNG. Each operation is therefore enclosed in `temporary_seed`. Calibration uses its own training-only stream; evaluation resets a different seed for every model, producing identical held-out samples without storing them.


In [ ]:
def make_synthetic_model(seed):
    if not FAST_DEV_RUN:
        snapshot = snapshot_download(
            repo_id=BENCHMARK_MODEL_ID, revision=BENCHMARK_REVISION
        )
        synthetic = SyntheticModel.load_from_disk(snapshot, device=str(DEVICE))
        assert (synthetic.cfg.hidden_dim, synthetic.cfg.num_features) == (
            hidden_dim,
            n_true_features,
        )
        return synthetic
    data_seed = data_model_seed_offset + seed
    with temporary_seed(data_seed):
        return SyntheticModel(
            SyntheticModelConfig(
                num_features=n_true_features,
                hidden_dim=hidden_dim,
                firing_probability=ConstantFiringProbabilityConfig(
                    scratch_support_density
                ),
                mean_firing_magnitudes=1.0,
                std_firing_magnitudes=0.5,
                bias=False,
                seed=data_seed,
            ),
            device=str(DEVICE),
        )


def raw_stream(synthetic, size):
    return SyntheticActivationIterator(
        feature_dict=synthetic.feature_dict,
        activations_generator=synthetic.activation_generator,
        batch_size=size,
    )


def calibrate_scale(synthetic, seed):
    scaler = OfficialActivationScaler()
    with temporary_seed(calibration_seed_offset + seed):
        scaler.estimate_scaling_factor(
            d_in=hidden_dim,
            data_provider=raw_stream(synthetic, calibration_batch_size),
            n_batches_for_norm_estimate=calibration_samples // calibration_batch_size,
        )
    assert scaler.scaling_factor is not None
    return ActivationScale(float(scaler.scaling_factor))


print(
    "mode:", "FAST 6x2 integration" if FAST_DEV_RUN else "full exploratory",
    "training samples:", total_training_samples,
    "evaluation samples:", eval_samples,
)


## Six native training specifications

The five baselines are direct official SAELens classes. VG uses its registered SAELens training class and constraint-preserving trainer. All receive the same externally scaled activation stream and keep internal normalization disabled.


In [ ]:
METHOD_ORDER = ("vgsae", "l1", "topk", "batchtopk", "jumprelu", "gated")
METHOD_LABELS = {
    "vgsae": "VG-SAE",
    "l1": "L1-ReLU",
    "topk": "TopK",
    "batchtopk": "BatchTopK",
    "jumprelu": "JumpReLU",
    "gated": "Gated",
}
METHOD_COLORS = dict(
    zip(METHOD_ORDER, ("tab:blue", "tab:orange", "tab:green", "tab:purple", "tab:brown", "tab:red"))
)


def build_specs():
    controls = {
        "vgsae": ("lambda_sparsity", gamma_values),
        "l1": ("l1_coefficient", l1_values),
        "topk": ("k", topk_values),
        "batchtopk": ("k", [float(k) for k in topk_values]),
        "jumprelu": ("l0_coefficient", jumprelu_values),
        "gated": ("l1_coefficient", l1_values),
    }
    return [
        {"method": method, "control_name": name, "control_value": value}
        for method in METHOD_ORDER
        for name, values in [controls[method]]
        for value in values
    ]


def build_model(spec):
    common = dict(
        d_in=hidden_dim,
        d_sae=d_sae,
        device=str(DEVICE),
        normalize_activations="none",
    )
    method, value = spec["method"], spec["control_value"]
    if method == "vgsae":
        return VGTrainingSAE(
            VGTrainingSAEConfig(
                **common,
                beta=beta,
                beta_mode=VG_BETA_MODE,
                lambda_sparsity=float(value),
            )
        )
    if method == "l1":
        return StandardTrainingSAE(
            StandardTrainingSAEConfig(**common, l1_coefficient=float(value))
        )
    if method == "topk":
        return TopKTrainingSAE(
            TopKTrainingSAEConfig(
                **common, k=int(value), rescale_acts_by_decoder_norm=True
            )
        )
    if method == "batchtopk":
        return BatchTopKTrainingSAE(
            BatchTopKTrainingSAEConfig(
                **common, k=float(value), rescale_acts_by_decoder_norm=True
            )
        )
    if method == "jumprelu":
        return JumpReLUTrainingSAE(
            JumpReLUTrainingSAEConfig(**common, l0_coefficient=float(value))
        )
    if method == "gated":
        return GatedTrainingSAE(
            GatedTrainingSAEConfig(**common, l1_coefficient=float(value))
        )
    raise ValueError(f"Unknown method: {method}")


In [ ]:
def trainer_config():
    return SAETrainerConfig(
        total_training_samples=total_training_samples,
        train_batch_size_samples=batch_size,
        lr=learning_rate,
        lr_end=learning_rate,
        device=str(DEVICE),
        dead_feature_window=dead_feature_window,
        feature_sampling_window=max(dead_feature_window, 2),
        n_checkpoints=0,
        save_final_checkpoint=False,
    )


def train_one(spec, synthetic, scale, seed):
    with temporary_seed(model_seed_offset + seed):
        model = build_model(spec)
    scaler = OfficialActivationScaler(scaling_factor=scale.factor)
    provider = map(scaler.scale, raw_stream(synthetic, batch_size))
    trainer_type = VGSAETrainer if spec["method"] == "vgsae" else SAETrainer
    trainer = trainer_type(cfg=trainer_config(), sae=model, data_provider=provider)
    started = time.perf_counter()
    with temporary_seed(train_seed_offset + seed):
        trained = trainer.fit()
    assert trainer.n_training_samples == total_training_samples
    return trained, time.perf_counter() - started


## Official inference export and evaluation

The generic conversion below uses each training config's registered inference architecture and `process_state_dict_for_saving_inference`. It then folds decoder norms. BatchTopK consequently evaluates as the official JumpReLU inference model learned from its running global threshold, not as a test-batch TopK.


In [ ]:
@torch.no_grad()
def official_metrics(model, synthetic, scale, seed):
    with temporary_seed(eval_seed_offset + seed):
        result = eval_sae_on_synthetic_data(
            sae=model,
            feature_dict=synthetic.feature_dict,
            activations_generator=synthetic.activation_generator,
            num_samples=eval_samples,
            batch_size=eval_batch_size,
            activation_scaler=OfficialActivationScaler(scale.factor),
        )
    return {
        "sae_l0": result.sae_l0,
        "true_l0": result.true_l0,
        "rho_model": result.sae_l0 / d_sae,
        "true_l0_over_d_sae": result.true_l0 / d_sae,
        "true_feature_density": result.true_l0 / n_true_features,
        "dead_latents": result.dead_latents,
        "dead_fraction": result.dead_latents / d_sae,
        "shrinkage": result.shrinkage,
        "explained_variance": result.explained_variance,
        "mcc": result.mcc,
        "uniqueness": result.uniqueness,
        "classification_precision": result.classification.precision,
        "classification_recall": result.classification.recall,
        "classification_f1": result.classification.f1_score,
        "classification_accuracy": result.classification.accuracy,
    }


@torch.no_grad()
def vg_expected_metrics(model, synthetic, scale, seed):
    empty = {
        "vg_posterior_rho": np.nan,
        "vg_expected_l0": np.nan,
        "vg_posterior_variance": np.nan,
        "vg_expected_explained_variance": np.nan,
        "vg_expected_relative_error": np.nan,
    }
    if not isinstance(model, VGSAE):
        return empty

    scaler = OfficialActivationScaler(scale.factor)
    variance = ExplainedVarianceCalculator()
    posterior_sum = posterior_variance_sum = squared_error = squared_input = 0.0
    processed = 0
    with temporary_seed(eval_seed_offset + seed):
        while processed < eval_samples:
            current = min(eval_batch_size, eval_samples - processed)
            true_features = synthetic.activation_generator.sample(current)
            hidden = synthetic.feature_dict(true_features)
            posterior = model.posterior(scaler.scale(hidden))
            expected = scaler.unscale(model.decode(posterior["expected_code"]))
            probabilities = posterior["m"]
            variance.add_batch(expected, hidden)
            posterior_sum += probabilities.sum().item()
            posterior_variance_sum += (probabilities * (1 - probabilities)).sum().item()
            squared_error += (expected - hidden).pow(2).sum().item()
            squared_input += hidden.pow(2).sum().item()
            processed += current
    return {
        "vg_posterior_rho": posterior_sum / (processed * d_sae),
        "vg_expected_l0": posterior_sum / processed,
        "vg_posterior_variance": posterior_variance_sum / (processed * d_sae),
        "vg_expected_explained_variance": variance.compute(),
        "vg_expected_relative_error": (squared_error / max(squared_input, 1e-12)) ** 0.5,
    }


## Run the 6×2 FAST integration or full sweep

Every result records scale, exact SAELens source identity, and all schedule seeds. Full-mode controls and sample counts are environment-overridable because the default five-seed grid is already expensive; a 200M-sample leaderboard reproduction is deliberately separate.


In [ ]:
rows = []
if RUN_EXPERIMENT:
    for seed in seeds:
        synthetic = make_synthetic_model(seed)
        scale = calibrate_scale(synthetic, seed)
        for spec in build_specs():
            run_id = (
                f"{spec['method']}_{spec['control_name']}="
                f"{spec['control_value']}_seed={seed}"
            )
            print("training", run_id)
            trained, elapsed = train_one(spec, synthetic, scale, seed)
            inference = to_inference_sae(trained, fold_decoder_norm=True)
            metrics = official_metrics(inference, synthetic, scale, seed)
            metrics.update(vg_expected_metrics(inference, synthetic, scale, seed))
            rows.append(
                {
                    "run_id": run_id,
                    "seed": seed,
                    "method": spec["method"],
                    "method_label": METHOD_LABELS[spec["method"]],
                    "control_name": spec["control_name"],
                    "control_value": spec["control_value"],
                    "inference_architecture": inference.cfg.architecture(),
                    "model_init_seed": model_seed_offset + seed,
                    "calibration_seed": calibration_seed_offset + seed,
                    "train_stream_seed": train_seed_offset + seed,
                    "eval_stream_seed": eval_seed_offset + seed,
                    "activation_scale": scale.factor,
                    "n_calibration_samples": calibration_samples,
                    "n_training_samples": total_training_samples,
                    "n_evaluation_samples": eval_samples,
                    "saelens_version": SAELENS_VERSION,
                    "saelens_commit": SAELENS_COMMIT,
                    "benchmark_revision": BENCHMARK_REVISION,
                    "train_seconds": elapsed,
                    **metrics,
                }
            )

results = pd.DataFrame(rows)
if not results.empty:
    assert len(results) == len(seeds) * len(build_specs())
    for column in (
        "activation_scale",
        "calibration_seed",
        "train_stream_seed",
        "eval_stream_seed",
        "true_l0",
    ):
        assert results.groupby("seed")[column].nunique().eq(1).all()
    assert results["n_training_samples"].eq(total_training_samples).all()
    results = results.sort_values(["method", "rho_model"]).reset_index(drop=True)
    results.to_csv(OUTPUT_DIR / "final_metrics.csv", index=False)
results


## Hard-rho comparison

The x-axis is official `sae_l0 / d_sae`. The dashed reference is `true_l0 / d_sae`, using the same denominator to mark equality of active counts. The generator's own `true_l0 / n_true_features` density is retained separately. VG posterior rho never replaces hard L0.


In [ ]:
if not results.empty:
    metrics = [
        ("explained_variance", r"hard $R^2$"),
        ("mcc", "MCC"),
        ("uniqueness", "uniqueness"),
        ("classification_f1", "classification F1"),
        ("dead_fraction", "dead fraction"),
        ("shrinkage", "shrinkage"),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(10, 5.5))
    for ax, (metric, label) in zip(axes.ravel(), metrics, strict=True):
        for method in METHOD_ORDER:
            subset = results[results.method == method].sort_values("rho_model")
            ax.plot(
                subset.rho_model,
                subset[metric],
                marker="o",
                linewidth=1,
                color=METHOD_COLORS[method],
                label=METHOD_LABELS[method],
            )
        ax.axvline(
            results.true_l0_over_d_sae.mean(),
            color="black",
            linestyle="--",
            linewidth=1,
        )
        ax.set(xlabel=r"hard L0 / $d_{sae}$", ylabel=label)
        ax.grid(alpha=0.25)
    axes[0, 0].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "hard_rho_official_metrics.png", dpi=160)
    plt.show()

    display_columns = [
        "method_label", "control_name", "control_value", "rho_model",
        "sae_l0", "true_l0", "explained_variance", "mcc", "uniqueness",
        "classification_f1", "vg_posterior_rho",
        "vg_expected_explained_variance", "activation_scale",
    ]
    display(results[display_columns])


## Interpretation checklist

- Compare official metrics at matched hard L0 density; the reference uses the same `d_sae` denominator.
- VG posterior rho and expected reconstruction are variational diagnostics, not hard firing metrics.
- BatchTopK uses its officially exported JumpReLU threshold at evaluation.
- Seed and scale columns audit common fresh training and identical held-out streams without caching support masks.
- FAST mode is integration smoke coverage. Publication claims require the full multi-seed protocol and an explicit compute budget.
